# Confidence calibration

How well the model's stated `confidence` matches how often it is right, for one `evaluate` sweep.

After https://arxiv.org/pdf/2512.16030

In [20]:
%matplotlib ipympl

In [21]:
import pandas as pd

from analysis.build_data import load_trace_df

In [22]:
run_name = "sweep-20261105-1621"

In [23]:
# one row per forecast of every child run of the sweep (see analysis/build_data.py)
forecasts = load_trace_df("outcome-confidence", run_name)

## Expected calibration error

ECE (Naeini et al., 2015) is the average gap between confidence and accuracy, with forecasts binned by their stated confidence into $B$ bins:

$$\mathrm{ECE} = \sum_{b=1}^{B} \frac{|B_b|}{N}\,\bigl|\mathrm{acc}(B_b) - \mathrm{conf}(B_b)\bigr|$$

Here a forecast is the outcome prompts' yes/no call, `confidence` is the model's stated probability that the call is right, and a forecast is accurate when the call matches `outcome_occurred`.

In [24]:
# scoreable forecasts: a yes/no call with a confidence, on a market that has resolved
calibrated = forecasts.dropna(
    subset=["predicted_outcome", "confidence", "outcome_occurred"]
)

In [25]:
calibrated["correct"] = calibrated["predicted_outcome"] == calibrated[
    "outcome_occurred"
].astype(bool)

In [26]:
B = 10

In [27]:
confidence_bins = pd.IntervalIndex.from_breaks([b / B for b in range(B + 1)])

In [28]:
calibrated["confidence_bin"] = pd.cut(
    calibrated["confidence"], confidence_bins, include_lowest=True
)

In [29]:
def expected_calibration_error(forecasts: pd.DataFrame) -> float:
    """sum_b |B_b|/N * |acc(B_b) - conf(B_b)| over the confidence bins."""
    bins = forecasts.groupby("confidence_bin", observed=True)
    acc = bins["correct"].mean()
    conf = bins["confidence"].mean()
    weight = bins.size() / len(forecasts)
    return (weight * (acc - conf).abs()).sum()

In [30]:
ece = expected_calibration_error(calibrated)

In [31]:
ece_by_model = calibrated.groupby("model").apply(
    expected_calibration_error, include_groups=False
)

In [32]:
# the reliability table behind the number: per bin, its size, conf(B_b) and acc(B_b)
reliability = calibrated.groupby("confidence_bin", observed=True).agg(
    n=("correct", "size"), conf=("confidence", "mean"), acc=("correct", "mean")
)

In [33]:
reliability

,n,conf,acc
confidence_bin,,,
"(0.2, 0.3]",6,0.248333,0.666667
"(0.3, 0.4]",12,0.353333,0.166667
"(0.4, 0.5]",13,0.465385,0.384615
"(0.5, 0.6]",43,0.550000,0.604651
"(0.6, 0.7]",27,0.664444,0.481481
"(0.7, 0.8]",28,0.748214,0.464286
"(0.8, 0.9]",23,0.874348,0.652174
"(0.9, 1.0]",48,0.960979,0.750000


In [34]:
ece_by_model

model
claude-haiku-4-5    0.21010
claude-opus-5-5     0.20027
dtype: float64